# 🌊 Week 3, Day 6 — June 6, 2026
## Document Master Schema & Save as Parquet



In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import os, json, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {"raw": BASE_DIR+"/data/raw","processed": BASE_DIR+"/data/processed",
        "metadata": BASE_DIR+"/data/metadata","visualizations": BASE_DIR+"/visualizations"}
LAT_MIN,LAT_MAX,LON_MIN,LON_MAX = 5,30,65,95
YEAR_MIN,YEAR_MAX = 2015,2026

In [ ]:
master = pd.read_csv(DIRS['processed']+'/master_table_v1.csv')
print(f'Master table loaded: {master.shape}')

In [ ]:
# Save as Parquet (much faster to read in downstream notebooks)
try:
    master.to_parquet(DIRS['processed']+'/master_table.parquet', index=False)
    print('Saved master_table.parquet ✅')
except Exception as e:
    print(f'Parquet save failed ({e}) — saving as optimized CSV instead')
    master.to_csv(DIRS['processed']+'/master_table_optimized.csv', index=False)
    print('Saved master_table_optimized.csv ✅')

In [ ]:
import datetime
schema_path = DIRS['metadata']+'/master_table_schema.txt'
with open(schema_path, 'w') as f:
    f.write('MASTER TABLE SCHEMA DOCUMENTATION\n')
    f.write(f'Generated: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}\n')
    f.write('='*60 + '\n\n')
    f.write(f'Shape: {master.shape[0]:,} rows × {master.shape[1]} columns\n\n')
    f.write('COLUMNS:\n')
    for col in master.columns:
        dtype = master[col].dtype
        nulls = master[col].isnull().sum()
        sample = str(master[col].dropna().iloc[0]) if nulls < len(master) else 'all null'
        f.write(f'  {col:<35} {str(dtype):<10} nulls={nulls:<6} sample={sample}\n')
    f.write('\nJOIN KEYS: grid_lat, grid_lon, year, month\n')
    f.write('SPATIAL CRS: EPSG:4326 (WGS84)\n')
    f.write('BOUNDING BOX: Lat 5N-30N, Lon 65E-95E\n')
    f.write('TIME WINDOW: 2015\n')
print(f'Schema documentation saved: {schema_path} ✅')
with open(schema_path) as f: print(f.read())